# Superstore Revenue & Profit Monitoring System

An automated business reporting pipeline for Superstore sales data.  
The system combines monthly sales files, validates data quality, calculates revenue and profitability KPIs, detects business risks, and exports stakeholder-ready reports and charts.

**Tools:** Python · pandas · matplotlib  
**Dataset:** Sample Superstore (`sample_-_superstore.xls`)  

---

## Configuration

All thresholds and paths are defined here. Edit this cell to customise alert sensitivity or folder locations without touching the rest of the notebook.

In [ ]:
# ── File paths ──────────────────────────────────────────────────────────────
DATA_FILE          = "sample_-_superstore.xls"
MONTHLY_FOLDER     = "monthly_sales_files"
CHARTS_FOLDER      = "charts"
OUTPUT_FOLDER      = "output_reports"

# ── Alert thresholds ─────────────────────────────────────────────────────────
HIGH_SALES_QUANTILE        = 0.75   # Top 25% of products by sales
LOW_PROFIT_MARGIN          = 0.05   # Below 5% margin = weak profitability
HIGH_DISCOUNT_THRESHOLD    = 0.30   # 30%+ average discount = risky
MONTHLY_DROP_THRESHOLD     = -10    # >10% MoM drop triggers an alert

## Imports

In [ ]:
import os
import glob

import pandas as pd
import matplotlib.pyplot as plt

## 1. Load Data

Load the raw Superstore XLS file. Make sure `sample_-_superstore.xls` is in the same directory as this notebook.

In [ ]:
df = pd.read_excel(DATA_FILE, engine="xlrd")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Null values:\n", df.isnull().sum())
df.head()

## 2. Data Preparation

Convert date columns to datetime format and engineer additional reporting fields:  
year, month, month-year period, and shipping duration in days.

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"]  = pd.to_datetime(df["Ship Date"])

df["Year"]          = df["Order Date"].dt.year
df["Month"]         = df["Order Date"].dt.month
df["Month_Year"]    = df["Order Date"].dt.to_period("M").astype(str)
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

print("Date range:", df["Order Date"].min(), "to", df["Order Date"].max())
print("Shape:", df.shape)

df[["Order Date", "Ship Date", "Year", "Month", "Month_Year", "Shipping Days"]].head()

### Split Into Monthly Files

The dataset is split into monthly CSV files to simulate a real reporting environment where new sales extracts arrive each month.

In [ ]:
os.makedirs(MONTHLY_FOLDER, exist_ok=True)

for month, month_df in df.groupby("Month_Year"):
    file_path = os.path.join(MONTHLY_FOLDER, f"sales_{month}.csv")
    month_df.to_csv(file_path, index=False)

print(f"Monthly CSV files created: {len(os.listdir(MONTHLY_FOLDER))}")

### Combine Monthly Files

All monthly CSVs are read and concatenated into a single dataframe, removing the need for manual copy-paste work.

In [ ]:
csv_files   = glob.glob(f"{MONTHLY_FOLDER}/*.csv")
combined_df = pd.concat(
    [pd.read_csv(f) for f in csv_files],
    ignore_index=True
)

# Re-parse dates after CSV round-trip
combined_df["Order Date"] = pd.to_datetime(combined_df["Order Date"])
combined_df["Ship Date"]  = pd.to_datetime(combined_df["Ship Date"])

print("Combined shape:", combined_df.shape)
combined_df.head()

## 3. Data Validation

Before calculating KPIs, the pipeline validates data reliability across six checks:  
missing columns, missing values, duplicate rows, negative sales, invalid discounts, and impossible shipping dates.

In [ ]:
required_columns = [
    "Order ID", "Order Date", "Ship Date", "Region", "Category",
    "Sub-Category", "Product Name", "Sales", "Quantity", "Discount", "Profit"
]

missing_columns = [col for col in required_columns if col not in combined_df.columns]

validation_checks = {
    "Missing required columns":        len(missing_columns),
    "Duplicate rows":                  combined_df.duplicated().sum(),
    "Negative sales records":          (combined_df["Sales"] < 0).sum(),
    "Zero / negative quantity records":(combined_df["Quantity"] <= 0).sum(),
    "Invalid discount records":        ((combined_df["Discount"] < 0) | (combined_df["Discount"] > 1)).sum(),
    "Invalid shipping dates":          (combined_df["Ship Date"] < combined_df["Order Date"]).sum(),
}

validation_summary = pd.DataFrame(
    validation_checks.items(),
    columns=["Check", "Result"]
)

validation_summary

## 4. KPI Calculation

Core business metrics: total sales, profit, order count, quantity, average order value, profit margin, and average discount.

In [ ]:
total_sales    = combined_df["Sales"].sum()
total_profit   = combined_df["Profit"].sum()
total_orders   = combined_df["Order ID"].nunique()
total_quantity = combined_df["Quantity"].sum()

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Sales", "Total Profit", "Total Orders",
        "Total Quantity Sold", "Average Order Value",
        "Profit Margin", "Average Discount"
    ],
    "Value": [
        total_sales,
        total_profit,
        total_orders,
        total_quantity,
        total_sales / total_orders,
        total_profit / total_sales,
        combined_df["Discount"].mean()
    ]
})

kpi_summary

### Monthly KPI Summary

Tracks performance over time with month-over-month sales and profit growth rates.

In [ ]:
monthly_kpi = combined_df.groupby("Month_Year").agg(
    total_sales  =("Sales",    "sum"),
    total_profit =("Profit",   "sum"),
    total_orders =("Order ID", "nunique"),
    total_quantity=("Quantity","sum"),
    avg_discount =("Discount", "mean")
).reset_index()

monthly_kpi["profit_margin"]       = monthly_kpi["total_profit"] / monthly_kpi["total_sales"]
monthly_kpi["average_order_value"] = monthly_kpi["total_sales"]  / monthly_kpi["total_orders"]
monthly_kpi["sales_growth_pct"]    = monthly_kpi["total_sales"].pct_change()  * 100
monthly_kpi["profit_growth_pct"]   = monthly_kpi["total_profit"].pct_change() * 100

monthly_kpi.head()

## 5. Business Risk Detection

Rule-based alerts flag potential risks that pure revenue reporting would miss.  
Each alert uses thresholds defined in the Configuration cell above.

In [ ]:
# Product-level aggregation
product_perf = combined_df.groupby(["Product Name", "Category", "Sub-Category"]).agg(
    total_sales  =("Sales",    "sum"),
    total_profit =("Profit",   "sum"),
    total_quantity=("Quantity","sum"),
    avg_discount =("Discount", "mean"),
    total_orders =("Order ID", "nunique")
).reset_index()

product_perf["profit_margin"] = product_perf["total_profit"] / product_perf["total_sales"]
product_perf.head()

### Alert 1 — High Sales, Low Profit

Products in the top 25% of sales but with a profit margin below 5%. These look successful on revenue but may be eroding profitability.

In [ ]:
high_sales_threshold = product_perf["total_sales"].quantile(HIGH_SALES_QUANTILE)

high_sales_low_profit = product_perf[
    (product_perf["total_sales"]    >= high_sales_threshold) &
    (product_perf["profit_margin"]  <  LOW_PROFIT_MARGIN)
].sort_values("total_sales", ascending=False)

print(f"Products flagged: {len(high_sales_low_profit)}")
high_sales_low_profit.head(10)

### Alert 2 — Loss-Making Products

Products where total profit is negative. These need pricing review, discount control, or removal from priority campaigns.

In [ ]:
loss_making_products = product_perf[
    product_perf["total_profit"] < 0
].sort_values("total_profit")

print(f"Loss-making products: {len(loss_making_products)}")
loss_making_products.head(10)

### Alert 3 — High Discount + Negative Profit

Products with an average discount of 30%+ that are still losing money. Indicates discount strategy is damaging profitability.

In [ ]:
high_discount_loss = product_perf[
    (product_perf["avg_discount"] >= HIGH_DISCOUNT_THRESHOLD) &
    (product_perf["total_profit"] <  0)
].sort_values("avg_discount", ascending=False)

print(f"High-discount loss products: {len(high_discount_loss)}")
high_discount_loss.head(10)

### Alert 4 — Category-Level Risk

Compares sales, profit, average discount, and profit margin across product categories.

In [ ]:
category_risk = combined_df.groupby("Category").agg(
    total_sales  =("Sales",    "sum"),
    total_profit =("Profit",   "sum"),
    avg_discount =("Discount", "mean"),
    total_quantity=("Quantity","sum")
).reset_index()

category_risk["profit_margin"] = category_risk["total_profit"] / category_risk["total_sales"]
category_risk.sort_values("profit_margin")

### Alert 5 — Region-Level Risk

Identifies markets with high sales but weak profitability or unusually high discount levels.

In [ ]:
region_risk = combined_df.groupby("Region").agg(
    total_sales  =("Sales",    "sum"),
    total_profit =("Profit",   "sum"),
    avg_discount =("Discount", "mean"),
    total_orders =("Order ID", "nunique")
).reset_index()

region_risk["profit_margin"] = region_risk["total_profit"] / region_risk["total_sales"]
region_risk.sort_values("profit_margin")

## 6. Monthly Performance Alerts

Flags months where sales, profit, or margin dropped more than 10% month-over-month.

In [ ]:
monthly_alerts = monthly_kpi.copy()
monthly_alerts["margin_change_pct"] = monthly_alerts["profit_margin"].pct_change() * 100

monthly_alerts["sales_alert"]  = monthly_alerts["sales_growth_pct"].apply(
    lambda x: "Sales Drop"  if x < MONTHLY_DROP_THRESHOLD else "Normal"
)
monthly_alerts["profit_alert"] = monthly_alerts["profit_growth_pct"].apply(
    lambda x: "Profit Drop" if x < MONTHLY_DROP_THRESHOLD else "Normal"
)
monthly_alerts["margin_alert"] = monthly_alerts["margin_change_pct"].apply(
    lambda x: "Margin Drop" if x < MONTHLY_DROP_THRESHOLD else "Normal"
)

risky_months = monthly_alerts[
    (monthly_alerts["sales_alert"]  != "Normal") |
    (monthly_alerts["profit_alert"] != "Normal") |
    (monthly_alerts["margin_alert"] != "Normal")
]

alert_summary = pd.DataFrame({
    "Alert Type": [
        "High sales but low profit products",
        "Loss-making products",
        "High discount loss products",
        "Risky months"
    ],
    "Count": [
        len(high_sales_low_profit),
        len(loss_making_products),
        len(high_discount_loss),
        len(risky_months)
    ]
})

alert_summary

## 7. Data Visualisation

Five charts saved to the `charts/` folder and displayed inline.

In [ ]:
os.makedirs(CHARTS_FOLDER, exist_ok=True)

### Monthly Sales Trend

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_kpi["Month_Year"], monthly_kpi["total_sales"], marker="o", color="#1f77b4")
plt.title("Monthly Sales Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{CHARTS_FOLDER}/monthly_sales_trend.png", dpi=150)
plt.show()

### Monthly Profit Trend

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_kpi["Month_Year"], monthly_kpi["total_profit"], marker="o", color="#2ca02c")
plt.title("Monthly Profit Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Total Profit ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{CHARTS_FOLDER}/monthly_profit_trend.png", dpi=150)
plt.show()

### Profit Margin by Category

In [ ]:
category_sorted = category_risk.sort_values("profit_margin", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(category_sorted["Category"], category_sorted["profit_margin"], color="#ff7f0e")
plt.title("Profit Margin by Category", fontsize=14, fontweight="bold")
plt.xlabel("Category")
plt.ylabel("Profit Margin")
plt.tight_layout()
plt.savefig(f"{CHARTS_FOLDER}/profit_margin_by_category.png", dpi=150)
plt.show()

### Sales vs Profit by Region

In [ ]:
region_sorted = region_risk.sort_values("total_sales", ascending=False)
x = range(len(region_sorted))

plt.figure(figsize=(9, 5))
plt.bar(x, region_sorted["total_sales"],   label="Sales",  color="#1f77b4", alpha=0.85)
plt.bar(x, region_sorted["total_profit"],  label="Profit", color="#2ca02c", alpha=0.85)
plt.xticks(x, region_sorted["Region"])
plt.title("Sales vs Profit by Region", fontsize=14, fontweight="bold")
plt.xlabel("Region")
plt.ylabel("Amount ($)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CHARTS_FOLDER}/sales_vs_profit_by_region.png", dpi=150)
plt.show()

### Top 10 Loss-Making Products

In [ ]:
top_losses = loss_making_products.head(10)

plt.figure(figsize=(12, 6))
plt.barh(top_losses["Product Name"], top_losses["total_profit"], color="#d62728")
plt.title("Top 10 Loss-Making Products", fontsize=14, fontweight="bold")
plt.xlabel("Total Profit ($)")
plt.ylabel("Product Name")
plt.tight_layout()
plt.savefig(f"{CHARTS_FOLDER}/top_loss_making_products.png", dpi=150)
plt.show()

## 8. Export Reports

All outputs are saved to `output_reports/` as CSV files for stakeholder use or dashboard ingestion.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# KPI tables
kpi_summary.to_csv(       f"{OUTPUT_FOLDER}/kpi_summary.csv",                index=False)
monthly_kpi.to_csv(       f"{OUTPUT_FOLDER}/monthly_kpi_summary.csv",        index=False)
validation_summary.to_csv(f"{OUTPUT_FOLDER}/data_validation_summary.csv",    index=False)

# Risk tables
high_sales_low_profit.to_csv(f"{OUTPUT_FOLDER}/high_sales_low_profit_products.csv", index=False)
loss_making_products.to_csv( f"{OUTPUT_FOLDER}/loss_making_products.csv",           index=False)
high_discount_loss.to_csv(   f"{OUTPUT_FOLDER}/high_discount_loss_products.csv",    index=False)
category_risk.to_csv(        f"{OUTPUT_FOLDER}/category_risk_summary.csv",          index=False)
region_risk.to_csv(          f"{OUTPUT_FOLDER}/region_risk_summary.csv",            index=False)
risky_months.to_csv(         f"{OUTPUT_FOLDER}/risky_months.csv",                   index=False)

# Combined cleaned data
combined_df.to_csv(f"{OUTPUT_FOLDER}/cleaned_combined_sales_data.csv", index=False)

print("All report files exported:")
print(*os.listdir(OUTPUT_FOLDER), sep="\n")

### Combined Business Risk Alert File

Merges product-level and month-level risks into a single exception report for stakeholders.

In [ ]:
alerts = []

for _, row in high_sales_low_profit.head(10).iterrows():
    alerts.append({
        "Alert Type": "High Sales Low Profit",
        "Entity":      row["Product Name"],
        "Category":    row["Category"],
        "Sales":       row["total_sales"],
        "Profit":      row["total_profit"],
        "Profit Margin": row["profit_margin"],
        "Reason":      "Product has high sales but weak profit margin"
    })

for _, row in loss_making_products.head(10).iterrows():
    alerts.append({
        "Alert Type": "Loss Making Product",
        "Entity":      row["Product Name"],
        "Category":    row["Category"],
        "Sales":       row["total_sales"],
        "Profit":      row["total_profit"],
        "Profit Margin": row["profit_margin"],
        "Reason":      "Product generated negative profit"
    })

for _, row in risky_months.iterrows():
    alerts.append({
        "Alert Type": "Monthly Performance Risk",
        "Entity":      row["Month_Year"],
        "Category":    "Overall Business",
        "Sales":       row["total_sales"],
        "Profit":      row["total_profit"],
        "Profit Margin": row["profit_margin"],
        "Reason":      f"{row['sales_alert']}, {row['profit_alert']}, {row['margin_alert']}"
    })

alerts_df = pd.DataFrame(alerts)
alerts_df.to_csv(f"{OUTPUT_FOLDER}/business_risk_alerts.csv", index=False)

print(f"Total alerts generated: {len(alerts_df)}")
alerts_df.head()

## 9. Automated Business Insights

Plain-language summaries generated programmatically from the analysis outputs.

In [ ]:
best_sales_month   = monthly_kpi.loc[monthly_kpi["total_sales"].idxmax()]
worst_sales_month  = monthly_kpi.loc[monthly_kpi["total_sales"].idxmin()]
best_profit_month  = monthly_kpi.loc[monthly_kpi["total_profit"].idxmax()]
worst_profit_month = monthly_kpi.loc[monthly_kpi["total_profit"].idxmin()]

best_category_sales     = category_risk.loc[category_risk["total_sales"].idxmax()]
weakest_category_margin = category_risk.loc[category_risk["profit_margin"].idxmin()]
best_region_sales       = region_risk.loc[region_risk["total_sales"].idxmax()]
weakest_region_margin   = region_risk.loc[region_risk["profit_margin"].idxmin()]

avg_disc_loss       = loss_making_products["avg_discount"].mean()
avg_disc_profitable = product_perf[product_perf["total_profit"] > 0]["avg_discount"].mean()

insights = [
    f"Highest sales month : {best_sales_month['Month_Year']}  (${best_sales_month['total_sales']:,.2f})",
    f"Lowest  sales month : {worst_sales_month['Month_Year']} (${worst_sales_month['total_sales']:,.2f})",
    f"Highest profit month: {best_profit_month['Month_Year']}  (${best_profit_month['total_profit']:,.2f})",
    f"Lowest  profit month: {worst_profit_month['Month_Year']} (${worst_profit_month['total_profit']:,.2f})",
    f"Highest-sales category : {best_category_sales['Category']}",
    f"Weakest-margin category: {weakest_category_margin['Category']}",
    f"Highest-sales region   : {best_region_sales['Region']}",
    f"Weakest-margin region  : {weakest_region_margin['Region']}",
    f"Avg discount — loss products: {avg_disc_loss:.2%}  vs  profitable: {avg_disc_profitable:.2%}",
    f"Monitoring detected {len(risky_months)} risky months and {len(loss_making_products)} loss-making products.",
]

for line in insights:
    print("-", line)

with open(f"{OUTPUT_FOLDER}/business_insights.txt", "w") as f:
    f.write("\n".join(f"- {line}" for line in insights))

print("\nInsights saved to output_reports/business_insights.txt")

## Conclusion

This notebook demonstrates an end-to-end automated reporting pipeline:

- **Data ingestion** — loads raw XLS and simulates monthly file delivery
- **Validation** — six-point data quality check before any KPIs are calculated
- **KPI reporting** — overall and monthly revenue, profit, margin, and growth metrics
- **Risk detection** — five rule-based alerts at product, category, region, and month level
- **Visualisation** — five production-ready charts saved to `charts/`
- **Export** — ten CSV reports and one plain-language insight summary in `output_reports/`

All thresholds are configurable in the **Configuration** cell at the top.